In [75]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

def scrape_turtlemint_table(url, insurer, policy_name):
    html = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}).text
    soup = BeautifulSoup(html, "html.parser")
    table = soup.find("table")

    hospital_names = []

    if table:
        rows = table.find_all("tr")[1:]  # skip header
        for row in rows:
            cols = row.find_all("td")
            if len(cols) >= 2:
                hospital_names.append(cols[1].get_text(strip=True))

    df = pd.DataFrame({
        "policy_name": policy_name,
        "insurer": insurer,
        "hospital_name": hospital_names,
        "city": "Pune",
        "coverage_type": "Cashless"
    })

    # Clean hospital names (join-safe)
    df["hospital_name_clean"] = (
        df["hospital_name"]
        .str.lower()
        .str.replace(r"[^a-z0-9 ]", "", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    return df


In [69]:
tata_df = scrape_turtlemint_table(
    "https://www.turtlemint.com/health-insurance/tata-aig/network-hospitals/pune/",
    insurer="TATA AIG",
    policy_name="TATA AIG MediCare"
)
tata_df.head()

,policy_name,insurer,hospital_name,city,coverage_type,hospital_name_clean
0,TATA AIG MediCare,TATA AIG,yogesh hosptial,Pune,Cashless,yogesh hosptial
1,TATA AIG MediCare,TATA AIG,deendayal hospital,Pune,Cashless,deendayal hospital
2,TATA AIG MediCare,TATA AIG,atharva netralaya and research pvt ltd,Pune,Cashless,atharva netralaya and research pvt ltd
3,TATA AIG MediCare,TATA AIG,oyster and pearl hospital pvt. ltd.,Pune,Cashless,oyster and pearl hospital pvt ltd
4,TATA AIG MediCare,TATA AIG,sancheti institute for orthopedics and rahabil...,Pune,Cashless,sancheti institute for orthopedics and rahabil...


In [70]:
care_df = scrape_turtlemint_table(
    "https://www.turtlemint.com/health-insurance/care/network-hospitals/pune/",
    insurer="Care Health",
    policy_name="Care Supreme"
)
care_df.head()

,policy_name,insurer,hospital_name,city,coverage_type,hospital_name_clean
0,Care Supreme,Care Health,poona laser centre,Pune,Cashless,poona laser centre
1,Care Supreme,Care Health,modern hospital,Pune,Cashless,modern hospital
2,Care Supreme,Care Health,dhanwantari hospital,Pune,Cashless,dhanwantari hospital
3,Care Supreme,Care Health,pramodini urology foundation,Pune,Cashless,pramodini urology foundation
4,Care Supreme,Care Health,naik hospital,Pune,Cashless,naik hospital


In [71]:
hdfc_df = scrape_turtlemint_table(
    "https://www.turtlemint.com/health-insurance/hdfc-ergo/network-hospitals/pune/",
    insurer="HDFC ERGO",
    policy_name="Optima Restore"
)
hdfc_df.head()

,policy_name,insurer,hospital_name,city,coverage_type,hospital_name_clean
0,Optima Restore,HDFC ERGO,surya hospital,Pune,Cashless,surya hospital
1,Optima Restore,HDFC ERGO,mjm hospital,Pune,Cashless,mjm hospital
2,Optima Restore,HDFC ERGO,gupte hospital of accurate diagnostics,Pune,Cashless,gupte hospital of accurate diagnostics
3,Optima Restore,HDFC ERGO,shree samarth hospital,Pune,Cashless,shree samarth hospital
4,Optima Restore,HDFC ERGO,kids clinic india pvt ltd,Pune,Cashless,kids clinic india pvt ltd


In [78]:
birla_df = scrape_turtlemint_table(
    "https://www.turtlemintinsurance.com/health-insurance/aditya-birla/network-hospitals/pune/",
    insurer="Aditya Birla",
    policy_name="Health Insurance Plan"
)
birla_df.head()

,policy_name,insurer,hospital_name,city,coverage_type,hospital_name_clean
0,Health Insurance Plan,Aditya Birla,king edwaroad memorial hospital,Pune,Cashless,king edwaroad memorial hospital
1,Health Insurance Plan,Aditya Birla,sancheti hospital for orthopedics and rehabili...,Pune,Cashless,sancheti hospital for orthopedics and rehabili...
2,Health Insurance Plan,Aditya Birla,surya hospital,Pune,Cashless,surya hospital
3,Health Insurance Plan,Aditya Birla,rajasthani & gujarati charitable foundation's ...,Pune,Cashless,rajasthani gujarati charitable foundations poo...
4,Health Insurance Plan,Aditya Birla,mjm hospital,Pune,Cashless,mjm hospital


In [80]:
import pandas as pd

# assuming you already have these DataFrames
# tata_df, care_df, hdfc_df, birla_df

# 1️⃣ Combine all
insurance_df = pd.concat([tata_df, care_df, hdfc_df, birla_df], ignore_index=True)

# 2️⃣ Deduplicate to avoid repeated hospitals for same insurer
insurance_df = insurance_df.drop_duplicates(subset=['insurer', 'hospital_name_clean'])

# 3️⃣ Optional: reset index
insurance_df.reset_index(drop=True, inplace=True)

# 4️⃣ Save to CSV
insurance_df.to_csv("insurance.csv", index=False)

# 5️⃣ Quick sanity check
print("Total rows:", len(insurance_df))
insurance_df.head(10)


Total rows: 508


,policy_name,insurer,hospital_name,city,coverage_type,hospital_name_clean
0,TATA AIG MediCare,TATA AIG,yogesh hosptial,Pune,Cashless,yogesh hosptial
1,TATA AIG MediCare,TATA AIG,deendayal hospital,Pune,Cashless,deendayal hospital
2,TATA AIG MediCare,TATA AIG,atharva netralaya and research pvt ltd,Pune,Cashless,atharva netralaya and research pvt ltd
3,TATA AIG MediCare,TATA AIG,oyster and pearl hospital pvt. ltd.,Pune,Cashless,oyster and pearl hospital pvt ltd
4,TATA AIG MediCare,TATA AIG,sancheti institute for orthopedics and rahabil...,Pune,Cashless,sancheti institute for orthopedics and rahabil...
5,TATA AIG MediCare,TATA AIG,national institute of ophthalmology,Pune,Cashless,national institute of ophthalmology
6,TATA AIG MediCare,TATA AIG,n.m.wadia institute of cardiology,Pune,Cashless,nmwadia institute of cardiology
7,TATA AIG MediCare,TATA AIG,karne hospital pvt ltd,Pune,Cashless,karne hospital pvt ltd
8,TATA AIG MediCare,TATA AIG,sharada clinic,Pune,Cashless,sharada clinic
9,TATA AIG MediCare,TATA AIG,deodhar eye hospital,Pune,Cashless,deodhar eye hospital


In [81]:
!pip install rapidfuzz

In [82]:
from google.colab import files
uploaded = files.upload()

Saving Hospitals.csv to Hospitals.csv


In [83]:
import pandas as pd

# Clean Pune hospitals
hospitals_df = pd.read_csv("Hospitals.csv")

# Insurance networks (the combined four insurers)
insurance_df = pd.read_csv("insurance.csv")

# Ensure hospital_name_clean exists in hospitals dataset
hospitals_df['hospital_name_clean'] = (
    hospitals_df['name']
    .str.lower()
    .str.replace(r"[^a-z0-9 ]", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)


In [84]:
from rapidfuzz import process, fuzz

# Function to match a single hospital name to insurance network
def match_insurance(hosp_name, insurance_names, limit=3):
    results = process.extract(
        hosp_name, insurance_names, scorer=fuzz.token_sort_ratio, limit=limit
    )
    # Keep only matches with >80% similarity
    return [match[0] for match in results if match[1] >= 80]

# Apply to all hospitals
insurance_names_list = insurance_df['hospital_name_clean'].tolist()
hospitals_df['matched_insurers'] = hospitals_df['hospital_name_clean'].apply(
    lambda x: match_insurance(x, insurance_names_list)
)


In [85]:
# Build a lookup dictionary from hospital_name_clean → insurer
hospital_to_insurer = insurance_df.groupby('hospital_name_clean')['insurer'].apply(list).to_dict()

# Map matched insurers
def get_insurer_names(matches):
    insurers = []
    for m in matches:
        insurers.extend(hospital_to_insurer.get(m, []))
    return list(set(insurers))  # unique insurers

hospitals_df['insurance_networks'] = hospitals_df['matched_insurers'].apply(get_insurer_names)


In [88]:
print(hospitals_df.columns.tolist())

['name', 'lat', 'long', 'facility_type', 'ownership', 'emergency_beds_x', 'ambulance_available_x', 'emergency_beds_y', 'ambulance_available_y', 'hospital_name_clean', 'insurance_networks']


In [89]:
# Drop temporary column
hospitals_df.drop(columns=['matched_insurers'], inplace=True, errors='ignore')

# Save merged dataset
hospitals_df.to_csv("hospitals_with_insurance.csv", index=False)

# Quick check
hospitals_df[['name','insurance_networks']].head(20)


,name,insurance_networks
0,Bharti Homeopathic Hospitalnan,[]
1,Bharti Hospitalnan,"[Aditya Birla, HDFC ERGO]"
2,Bharti Ayurved Hospitalnan,[]
3,Yugpurush Raja Shiv Chatrapati Bibvewadi (Appa...,[]
4,Trimurti Hospital,[]
5,Eye Clinic,"[TATA AIG, Aditya Birla]"
6,Adate General Hospital,[]
7,Sidhhi Hospital Laproscopy Center,[Care Health]
8,Rakshak Multi Specility Hospital,[]
9,rejuva skin clinic,[]
